# 🏍️ Entrenamiento - Detector de motos y cabezas

**ACELERADOR RECOMENDADO:** GPU T4 x2

**DEBE ESTAR SUBIDO Y ASOCIADO EL DATASET CORRESPONDIENTE:**
Este script buscará automáticamente el dataset en `/kaggle/input/`

In [ ]:
MODEL_NAME = 'detector_v4'

## 1. Instalación de Ultralytics

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configurar correctamente el Dataset
Como los datasets subidos son de solo lectura (`/kaggle/input/`), buscamos el archivo `data.yaml`, lo copiamos a la carpeta de trabajo temporal (`/kaggle/working/`), y arreglamos las rutas absolutas para que YOLO encuentre las imágenes.

In [ ]:
import os
import glob
import yaml
import shutil
from pathlib import Path

input_dir = Path('/kaggle/input')
yaml_files = list(input_dir.glob('**/*.yaml'))
if not yaml_files:
    raise FileNotFoundError("No se encontró ningún archivo .yaml en /kaggle/input/")
original_yaml = yaml_files[0]
dataset_root = original_yaml.parent
print(f"✅ Dataset encontrado en: {dataset_root}")

work_dir = Path('/kaggle/working/')
new_yaml_path = work_dir / 'data_fixed.yaml'
with open(original_yaml, 'r') as f:
    config = yaml.safe_load(f)
config['path'] = str(dataset_root)
with open(new_yaml_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print(f"✅ Archivo YAML modificado y guardado en: {new_yaml_path}")

## 3. Entrenamiento

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt') 

results = model.train(
    data=str(new_yaml_path),
    epochs=150,
    patience=50,
    imgsz=1024,           # CLAVE para recall de objetos pequeños
    batch=16,             # Aprovechamos los 16GB de la GPU T4
    device=[0, 1],        # Usa las 2 GPUs (T4 x2) simultáneamente
    freeze=10,            # Congela el backbone (más veloz, menos RAM)
    amp=True,             # Mixed Precision
    
    # OPTIMIZADOR
    optimizer='SGD',      # SGD con momentum suele lograr mayor mAP en detección
    lr0=0.001,             # Learning rate base para SGD en YOLO
    lrf=0.01,             # Cae hasta el 1% del inicial (0.0001)
    momentum=0.937,       # Inercia estándar de YOLO para no atascarse
    weight_decay=0.0005,  # Penaliza pesos muy altos (anti-overfitting)
    warmup_epochs=3.0,    # 3 epochs de calentamiento para estabilizar gradientes iniciales

    # AUGMENTATIONS
    hsv_h=0.015,          # Variación de hue ±1.5% (cambios sutiles luz solar)
    hsv_s=0.7,            # Variación de saturación (días nublados vs soleados)
    hsv_v=0.4,            # Variación de valor/brillo (sombras, contraluz)
    degrees=5.0,          # Rotación ±5° (cámaras levemente inclinadas o vibración)
    translate=0.1,        # Traslación ±10% (pequeños movimientos del encuadre)
    scale=0.5,            # Zoom in/out hasta 50% (motos a distintas distancias)
    shear=2.0,            # Cizallamiento leve ±2° (perspectiva de la cámara)
    perspective=0.0,      # NO usar transformación 3D (cámara fija, no aporta)
    flipud=0.0,           # NO voltear verticalmente (motos circulan 'hacia abajo')
    fliplr=0.5,           # Voltear horizontal 50% (motos de ambos lados)
    mosaic=0.5,           # Reducido (no rompe motos grandes, ayuda a cascos)
    mixup=0.0,            # Apagado (evita motos semi-transparentes)
    copy_paste=0.1,       # Pegar objetos en fondos aleatorios
    
    # NMS / VALIDACIÓN
    conf=0.001,           # Umbral NMS VALIDACIÓN (maximizar recall en mAP)
    iou=0.6,              # IoU threshold para NMS
    max_det=20,           # Máx. detecciones (tráfico urbano)
    
    # --------------------------------------------------------
    # REGULARIZACIÓN ADICIONAL
    # --------------------------------------------------------
    label_smoothing=0.05, # Suavizado (reduce sobreconfianza)
    dropout=0.0,          # Dropout en la cabeza (apagado por defecto)
    
    project='/kaggle/working/runs',
    name=MODEL_NAME
)

## 4. Evaluación final en el set de Test
Evaluamos el mejor modelo obtenido sobre el conjunto de test aislado.

In [ ]:
from ultralytics import YOLO
import os
import glob
from IPython.display import Image, display

print("Evaluando en el set de TEST ambos modelos (best.pt y last.pt)...")
for weight_name in ['best.pt', 'last.pt']:
    weight_path = f'/kaggle/working/runs/{MODEL_NAME}/weights/{weight_name}'
    if not os.path.exists(weight_path):
        print(f"\n⚠️ No se encontró {weight_name}, omitiendo...")
        continue
        
    print(f"\n==========================================")
    print(f" Cargando modelo para evaluación: {weight_name}")
    print(f"==========================================")
    model = YOLO(weight_path)
    eval_name = f'test_eval_{weight_name.replace(".pt", "")}'
    
    test_results = model.val(data=str(new_yaml_path), split='test', project=f'/kaggle/working/runs/{MODEL_NAME}', name=eval_name)
    
    try:
        mAP50 = test_results.results_dict['metrics/mAP50(B)']
        mAP50_95 = test_results.results_dict['metrics/mAP50-95(B)']
        print(f"mAP@50:    {mAP50:.4f}")
        print(f"mAP@50-95: {mAP50_95:.4f}")
        
        # --- GUARDAR EN ARCHIVO PARA EL ZIP ---
        eval_dir = f'/kaggle/working/runs/{MODEL_NAME}/{eval_name}'
        os.makedirs(eval_dir, exist_ok=True)
        report_path = os.path.join(eval_dir, f'metricas_detalladas_{weight_name.replace(".pt", "")}.txt')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f'==========================================\n')
            f.write(f'   MÉTRICAS FINALES - DETECTOR ({weight_name})\n')
            f.write(f'==========================================\n')
            f.write(f'mAP@50:    {mAP50:.4f}\n')
            f.write(f'mAP@50-95: {mAP50_95:.4f}\n\n')
            f.write('Métricas completas y por clase:\n')
            f.write('-' * 50 + '\n')
            for key, value in test_results.results_dict.items():
                f.write(f'{key}: {value}\n')
        print(f"\n✅ Métricas detalladas guardadas en texto en: {report_path}")
        
    except Exception as e:
        print("Métricas:")
        print(test_results.results_dict)
        
    # Mostrar gráficos generados por YOLO en el Notebook
    image_files = glob.glob(f'{eval_dir}/*.png')
    for img_path in image_files:
        if any(name in img_path for name in ['PR_curve', 'confusion_matrix', 'results', 'F1_curve', 'normalized']):
            display(Image(filename=img_path, width=700))


## 5. Exportar y Descargar el Mejor Modelo

In [ ]:
import shutil

folder_to_zip = f'/kaggle/working/runs/{MODEL_NAME}'
zip_name = f'/kaggle/working/{MODEL_NAME}'

shutil.make_archive(zip_name, 'zip', folder_to_zip)
print("✅ Entrenamiento terminado. Ve a la pestaña 'Output' a la derecha y descarga el archivo f'{MODEL_NAME}.zip'.")